# 03d — SHAP Global và Dependence trên mô hình cuối

**Mục đích:** Tài liệu này chạy lại phân tích giải thích toàn cục (`03_shap.ipynb`) và phân tích dependence, tương tác, nhóm đặc trưng (`03_shap_dependence.ipynb`) trên mô hình ứng viên chính **LightGBM tuned + monotonic**. Hai notebook trên được thực hiện với XGBoost tham số mặc định, trước khi có kết quả tuning, và đều ghi nhận ở mục Giới hạn rằng phân tích cần được chạy lại trên mô hình cuối. Giải thích cục bộ của cùng mô hình nằm tại `03_shap_local.ipynb`.

**Tài liệu tham chiếu:** `reports/project_plan.md` (mục 3.5e), `reports/week2_review.md`, `reports/experiments_optuna_tuning.md`, `configs/tuned_params.yaml`, `configs/config.yaml` (khóa `feature_groups`).

**Thiết lập:**

| Hạng mục | Giá trị |
|---|---|
| Mô hình | `lightgbm_monotone` — `src.tune.fit_tuned_pipeline`, fit trên toàn bộ Train, không có SEX |
| Mẫu giải thích | 3.000 dòng lấy ngẫu nhiên từ tập Valid (`random_state = 42`), trùng với mẫu của `03_shap.ipynb` |
| Explainer | `shap.TreeExplainer`, `feature_perturbation = "tree_path_dependent"` |
| Thang giá trị SHAP | Log-odds (margin) của mô hình chưa hiệu chuẩn |
| Đối chiếu | `xgboost_default` trên cùng mẫu (mục 5) |

Tập Test không được sử dụng. Notebook được kiểm tra với `STUDY = "lightgbm_monotone"`.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import load_config  # noqa: E402
from src.data_split import load_split_data  # noqa: E402
from src.explain import aggregate_by_group, aggregate_by_variable, load_feature_groups, source_variable  # noqa: E402
from src.features import build_features  # noqa: E402
from src.pipelines import make_pipeline  # noqa: E402
from src.train import get_advanced_default_models  # noqa: E402
from src.tune import fit_tuned_pipeline  # noqa: E402

CONFIG = load_config()
SEED = CONFIG["random_state"]
STUDY = "lightgbm_monotone"
MONOTONE = set(CONFIG["tuning"]["monotone_increasing"])
N_SAMPLE = 3000
TOP_N = 9
MAX_DISPLAY = 20
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
PREFIX = f"shap_{STUDY}"

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 160)
# shap 0.46 cảnh báo thay đổi định dạng đầu ra của LightGBM nhị phân; không ảnh hưởng kết quả.
warnings.filterwarnings("ignore", category=UserWarning, module="shap")
print("shap", shap.__version__)

## 1. Mô hình và mẫu giải thích

Mô hình được fit trên toàn bộ tập Train bằng `fit_tuned_pipeline`, cùng tham số và `random_state` với mô hình đã đăng ký trên MLflow, nên notebook chạy lại được từ bản clone sạch mà không phụ thuộc thư mục `mlruns/`. Dữ liệu đưa vào explainer là đầu ra của bước tiền xử lý; các cột số được đưa về thang gốc bằng `inverse_transform` của `StandardScaler` chỉ để hiển thị.

In [ ]:
train_df, valid_df, _ = load_split_data()
X_train, y_train = build_features(train_df)
X_valid, y_valid = build_features(valid_df)

pipe = fit_tuned_pipeline(STUDY, X_train, y_train)
prep, model = pipe.named_steps["preprocess"], pipe.named_steps["clf"]
col_tf = prep.named_steps["preprocessor"]
columns = [name.split("__", 1)[1] for name in col_tf.get_feature_names_out()]
cat_vars = list(col_tf.transformers_[0][2])
num_cols = list(col_tf.transformers_[1][2])
n_cat_out = len(columns) - len(num_cols)

X_sample = X_valid.sample(N_SAMPLE, random_state=SEED)
X_model = prep.transform(X_sample)
X_display = pd.DataFrame(X_model.copy(), columns=columns, index=X_sample.index)
X_display[num_cols] = col_tf.named_transformers_["numeric"]["scaler"].inverse_transform(X_model[:, n_cat_out:])

print(f"ROC-AUC trên Valid (tham chiếu): {roc_auc_score(y_valid, pipe.predict_proba(X_valid)[:, 1]):.4f}")
print(f"Mẫu: {len(X_sample)} dòng | tỷ lệ vỡ nợ: {y_valid.loc[X_sample.index].mean():.3f}")
print(f"Số cột sau tiền xử lý: {len(columns)} ({n_cat_out} cột one-hot từ {len(cat_vars)} biến phân loại, {len(num_cols)} biến số)")

## 2. Giá trị SHAP

Tính cộng tính được kiểm tra trên thang margin: $\phi_0 + \sum_i \phi_i = m(x)$, với $m(x) = \text{logit}(\hat p)$ của mô hình.

In [ ]:
explainer = shap.TreeExplainer(model)
t0 = time.perf_counter()
phi = explainer.shap_values(X_model)
phi = phi[1] if isinstance(phi, list) else phi
elapsed = time.perf_counter() - t0
base = float(np.ravel(explainer.expected_value)[-1])

proba = model.predict_proba(X_model)[:, 1]
margin = np.log(proba / (1 - proba))
max_err = np.abs(base + phi.sum(axis=1) - margin).max()

print(f"Shape SHAP: {phi.shape} | thời gian: {elapsed:.2f} s")
print(f"Base value: {base:.4f} (margin) ≈ PD {1 / (1 + np.exp(-base)):.3f}")
print(f"Sai số cộng tính lớn nhất: {max_err:.2e}")
assert max_err < 1e-6

## 3. Mức độ quan trọng toàn cục

### 3.1. Summary Plot (beeswarm)

Mỗi điểm là một khách hàng trong mẫu; trục hoành là giá trị SHAP (log-odds), màu thể hiện giá trị của cột (đỏ: cao, xanh: thấp). Biểu đồ hiển thị 20 cột có mean(|SHAP|) cao nhất.

In [ ]:
sv = shap.Explanation(values=phi, base_values=np.full(len(phi), base), data=X_display.to_numpy(), feature_names=columns)
shap.plots.beeswarm(sv, max_display=MAX_DISPLAY, show=False)
plt.title(f"SHAP Summary Plot — {STUDY} (mẫu Valid, n = {N_SAMPLE:,})".replace(",", "."))
plt.tight_layout()
plt.savefig(FIG_DIR / f"{PREFIX}_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Số cột có mean(|SHAP|) = 0: {int((np.abs(phi).mean(axis=0) == 0).sum())}/{len(columns)}")

### 3.2. Theo biến gốc

Giá trị SHAP của các cột one-hot được cộng về biến gốc trước khi lấy trị tuyệt đối (`aggregate_by_variable`). Bảng này là cơ sở xếp hạng mức độ quan trọng; cột `monotonic` đánh dấu 11 biến có ràng buộc đơn điệu tăng.

In [ ]:
shap_by_var = aggregate_by_variable(phi, columns, cat_vars)
variables = list(shap_by_var.columns)
importance = shap_by_var.abs().mean().sort_values(ascending=False)

imp_table = importance.rename("mean_abs_shap").to_frame()
imp_table["share"] = imp_table["mean_abs_shap"] / imp_table["mean_abs_shap"].sum()
imp_table["cum_share"] = imp_table["share"].cumsum()
imp_table["rank"] = np.arange(1, len(imp_table) + 1)
imp_table["monotonic"] = imp_table.index.isin(MONOTONE)
imp_table.to_csv(FIG_DIR / f"{PREFIX}_importance_by_variable.csv")

sv_var = shap.Explanation(values=shap_by_var.to_numpy(), base_values=np.full(len(phi), base), feature_names=variables)
shap.plots.bar(sv_var, max_display=MAX_DISPLAY, show=False)
plt.title(f"mean(|SHAP|) theo biến gốc — {STUDY}")
plt.tight_layout()
plt.savefig(FIG_DIR / f"{PREFIX}_bar_by_variable.png", dpi=150, bbox_inches="tight")
plt.show()

display(imp_table.head(MAX_DISPLAY))
imp_table.loc[["AGE_BIN", "EDUCATION", "MARRIAGE"], ["mean_abs_shap", "share", "rank"]]

### 3.3. Theo nhóm đặc trưng

Giá trị SHAP của các biến trong cùng nhóm (`configs/config.yaml`, khóa `feature_groups`) được cộng trước khi lấy trị tuyệt đối, đúng đại lượng dùng cho reason codes. Cột `Σ mean|SHAP| từng biến` cộng mức quan trọng của từng biến; `tỷ lệ triệt tiêu` đo mức các biến trong nhóm mang dấu ngược nhau trên cùng một khách hàng.

In [ ]:
groups = load_feature_groups()
label_of = {key: spec["label"] for key, spec in groups.items()}
shap_by_group = aggregate_by_group(shap_by_var, groups)

err = np.abs(base + shap_by_group.sum(axis=1) - margin).max()
print(f"Sai số cộng tính cấp nhóm: {err:.2e}")
assert err < 1e-6

group_importance = shap_by_group.abs().mean().sort_values(ascending=False)
sum_individual = pd.Series({label_of[k]: importance[spec["variables"]].sum() for k, spec in groups.items()})
group_table = pd.DataFrame({
    "mean|SHAP| nhóm": group_importance,
    "share": group_importance / group_importance.sum(),
    "Σ mean|SHAP| từng biến": sum_individual[group_importance.index],
})
group_table["tỷ lệ triệt tiêu"] = 1 - group_table["mean|SHAP| nhóm"] / group_table["Σ mean|SHAP| từng biến"]
group_table.to_csv(FIG_DIR / f"{PREFIX}_importance_by_group.csv")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(group_table.index[::-1], group_table["mean|SHAP| nhóm"][::-1], color="#d6336c")
for i, v in enumerate(group_table["mean|SHAP| nhóm"][::-1]):
    ax.text(v + 0.005, i, f"{v:.2f}", va="center")
ax.set_xlabel("mean(|SHAP nhóm|), log-odds")
ax.set_title(f"Mức độ quan trọng theo nhóm đặc trưng — {STUDY}")
fig.tight_layout()
fig.savefig(FIG_DIR / f"{PREFIX}_bar_by_group.png", dpi=150, bbox_inches="tight")
plt.show()
group_table

### 3.4. Độ ổn định theo mẫu

Thứ hạng theo biến gốc và theo nhóm được tính lại trên hai mẫu Valid khác (`random_state` = 0 và 1) và một mẫu 3.000 dòng từ tập Train, so với mẫu chính bằng hệ số Spearman, mức trùng Top 10 biến và thứ tự năm nhóm.

In [ ]:
ref_top10 = set(importance.index[:10])
rows = {}
for label, X_raw in {
    "Valid, seed 0": X_valid.sample(N_SAMPLE, random_state=0),
    "Valid, seed 1": X_valid.sample(N_SAMPLE, random_state=1),
    "Train, seed 42": X_train.sample(N_SAMPLE, random_state=SEED),
}.items():
    by_var = aggregate_by_variable(explainer.shap_values(prep.transform(X_raw)), columns, cat_vars)
    other = by_var.abs().mean()[importance.index]
    other_group = aggregate_by_group(by_var, groups).abs().mean()[group_importance.index]
    rows[label] = {
        "Spearman (biến)": importance.corr(other, method="spearman"),
        "Trùng Top 10": len(ref_top10 & set(other.sort_values(ascending=False).index[:10])),
        "Spearman (nhóm)": group_importance.corr(other_group, method="spearman"),
        "Cùng thứ tự nhóm": bool((other_group.sort_values(ascending=False).index == group_importance.index).all()),
    }
pd.DataFrame(rows).T

## 4. Dependence và tương tác

Tương tác được đo bằng `shap_interaction_values`, gộp về biến gốc trên cả hai chiều. Cường độ tương tác của cặp $(v, w)$ là trung bình trị tuyệt đối tổng các phần tử ma trận thuộc cặp đó, nhân 2 do ma trận đối xứng; phần đường chéo là hiệu ứng chính.

In [ ]:
t0 = time.perf_counter()
phi_int = explainer.shap_interaction_values(X_model)
phi_int = phi_int[1] if isinstance(phi_int, list) else phi_int
print(f"Interaction: {phi_int.shape} | thời gian: {time.perf_counter() - t0:.1f} s")
print(f"Sai số cộng tính (interaction): {np.abs(base + phi_int.sum((1, 2)) - margin).max():.2e}")

var_of_col = np.array([source_variable(c, cat_vars) for c in columns])
indicator = np.stack([(var_of_col == v) for v in variables], axis=1).astype(float)
phi_int_var = np.einsum("nab,av,bw->nvw", phi_int, indicator, indicator)
assert np.allclose(phi_int_var.sum(2), shap_by_var.to_numpy(), atol=1e-6)

strength = pd.DataFrame(np.abs(phi_int_var).mean(0) * 2, index=variables, columns=variables)
main_effect = pd.Series(np.diag(strength.to_numpy()) / 2, index=variables)
np.fill_diagonal(strength.values, 0.0)

pairs = (
    strength.where(np.triu(np.ones(strength.shape, dtype=bool), k=1))
    .stack()
    .sort_values(ascending=False)
    .rename("mean_abs_interaction")
    .to_frame()
)
pairs.index.names = ["biến 1", "biến 2"]
pairs.to_csv(FIG_DIR / f"{PREFIX}_interaction_pairs.csv")

top_vars = list(importance.index[:TOP_N])
partner = strength.idxmax(axis=1)
display(pairs.head(15))
pd.DataFrame({
    "mean|SHAP|": importance[top_vars],
    "hiệu ứng chính": main_effect[top_vars],
    "đối tác mạnh nhất": partner[top_vars],
    "cường độ": [strength.loc[v, partner[v]] for v in top_vars],
    "monotonic": [v in MONOTONE for v in top_vars],
})

### 4.1. Dependence Plot cho Top 9 biến

Trục hoành là giá trị gốc của biến, trục tung là đóng góp SHAP của biến (đã gộp one-hot). Màu thể hiện giá trị của biến tương tác mạnh nhất; đường đen là trung vị SHAP theo từng giá trị (biến rời rạc) hoặc theo phân vị (biến liên tục). Tiêu đề đánh dấu `[mono]` cho biến có ràng buộc đơn điệu tăng.

In [ ]:
def raw_value(var):
    # Cột số lấy từ X_display; PAY_1 lấy từ cột PAY_0 của dữ liệu thô (đổi tên ở bước tiền xử lý).
    if var in X_display.columns:
        return X_display[var].to_numpy()
    if var == "PAY_1":
        return X_sample["PAY_0"].to_numpy()
    return X_sample[var].to_numpy()


def trend(x, y, n_bins=10):
    df = pd.DataFrame({"x": x, "y": y})
    if df["x"].nunique() <= 15:
        g = df.groupby("x")["y"].median()
        return g.index.to_numpy(), g.to_numpy()
    df["b"] = pd.qcut(df["x"], n_bins, duplicates="drop")
    g = df.groupby("b", observed=True).agg(x=("x", "median"), y=("y", "median"))
    return g["x"].to_numpy(), g["y"].to_numpy()


fig, axes = plt.subplots(3, 3, figsize=(17, 13))
for ax, var in zip(axes.ravel(), top_vars):
    x, y = raw_value(var), shap_by_var[var].to_numpy()
    c = raw_value(partner[var])
    sc = ax.scatter(x, y, c=c, cmap="coolwarm", s=7, alpha=0.7, vmin=np.percentile(c, 2), vmax=np.percentile(c, 98))
    tx, ty = trend(x, y)
    ax.plot(tx, ty, color="black", lw=1.6)
    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.set_xlabel(var)
    ax.set_ylabel("SHAP (log-odds)")
    ax.set_title(f"{var} (hạng {top_vars.index(var) + 1})" + (" [mono]" if var in MONOTONE else ""))
    fig.colorbar(sc, ax=ax, label=partner[var])
fig.suptitle(f"SHAP Dependence Plot — {STUDY}, Top 9 biến (màu: đối tác tương tác mạnh nhất)", y=1.0)
fig.tight_layout()
fig.savefig(FIG_DIR / f"{PREFIX}_dependence_top9.png", dpi=130, bbox_inches="tight")
plt.show()

### 4.2. `PAY_1` và đóng góp của nhóm lịch sử trễ hạn

`PAY_1`–`PAY_6` được mã hóa one-hot nên không áp được ràng buộc đơn điệu. Bảng dưới đây so sánh, theo từng mức `PAY_1`, đóng góp của riêng `PAY_1` với đóng góp ròng của cả nhóm lịch sử trễ hạn (đại lượng dùng cho reason codes).

In [ ]:
delay_label = label_of["lich_su_tre_han"]
pay1_table = (
    pd.DataFrame({
        "PAY_1": raw_value("PAY_1"),
        "SHAP PAY_1": shap_by_var["PAY_1"].to_numpy(),
        "SHAP nhóm trễ hạn": shap_by_group[delay_label].to_numpy(),
        "PD": proba,
    })
    .groupby("PAY_1")
    .agg(**{
        "số khách hàng": ("PD", "size"),
        "SHAP PAY_1 (trung vị)": ("SHAP PAY_1", "median"),
        "SHAP nhóm trễ hạn (trung vị)": ("SHAP nhóm trễ hạn", "median"),
        "PD trung bình": ("PD", "mean"),
    })
)
pay1_table

## 5. So sánh với XGBoost mặc định

XGBoost tham số mặc định (cấu hình của `03_shap.ipynb` và `03_shap_dependence.ipynb`) được fit lại và giải thích trên cùng mẫu 3.000 dòng. So sánh ở đây chỉ nhằm xác định các kết luận nào của hai notebook trước còn giữ được trên mô hình cuối; phân tích mức đồng thuận đầy đủ giữa XGBoost, LightGBM và Logistic thuộc Tuần 3 – T2.

In [ ]:
pipe_xgb = make_pipeline(clone(get_advanced_default_models(SEED)["xgboost_default"])).fit(X_train, y_train)
prep_xgb = pipe_xgb.named_steps["preprocess"]
columns_xgb = [n.split("__", 1)[1] for n in prep_xgb.named_steps["preprocessor"].get_feature_names_out()]
assert columns_xgb == columns

phi_xgb = shap.TreeExplainer(pipe_xgb[-1]).shap_values(prep_xgb.transform(X_sample))
by_var_xgb = aggregate_by_variable(phi_xgb, columns, cat_vars)
importance_xgb = by_var_xgb.abs().mean()[importance.index]
group_xgb = aggregate_by_group(by_var_xgb, groups).abs().mean()

rank_xgb = importance_xgb.rank(ascending=False).astype(int)
print(f"Spearman thứ hạng biến (LightGBM vs XGBoost): {importance.corr(importance_xgb, method='spearman'):.3f}")
print(f"Trùng Top 10: {len(set(importance.index[:10]) & set(importance_xgb.sort_values(ascending=False).index[:10]))}/10")

display(pd.DataFrame({
    "hạng LightGBM": np.arange(1, 11),
    "hạng XGBoost": rank_xgb[importance.index[:10]].to_numpy(),
}, index=importance.index[:10]))

compare_group = pd.DataFrame({
    "share LightGBM": group_importance / group_importance.sum(),
    "share XGBoost": group_xgb[group_importance.index] / group_xgb.sum(),
})
compare_group["hạng LightGBM"] = np.arange(1, len(compare_group) + 1)
compare_group["hạng XGBoost"] = compare_group["share XGBoost"].rank(ascending=False).astype(int)
compare_group.to_csv(FIG_DIR / f"{PREFIX}_vs_xgboost_by_group.csv")
compare_group

## 6. Nhận xét

### 6.1. Tính đúng đắn và chi phí tính toán

- Sai số cộng tính lớn nhất ở mức $10^{-14}$ ở cả cấp cột, cấp nhóm và ma trận tương tác.
- Base value $\phi_0 = -1{,}562$ (margin), tương ứng PD ≈ 0,173, trùng với `03_shap_local.ipynb` do cùng mô hình. Giá trị này thấp hơn tỷ lệ vỡ nợ của Train (≈ 0,221) vì là trung bình trên thang log-odds.
- ROC-AUC trên Valid (tham chiếu) đạt 0,7903, nhất quán với CV (0,7906 ± 0,0074).
- Thời gian tính SHAP cho 3.000 dòng khoảng 2,5 giây; `shap_interaction_values` khoảng 45 giây (XGBoost: khoảng 220 giây theo `03_shap_dependence.ipynb`). Tương tác vẫn không phù hợp để tính trực tiếp trên demo.

### 6.2. Mức độ quan trọng toàn cục

- **Lịch sử trễ hạn chiếm ưu thế rõ rệt.** Bốn biến đầu đều thuộc nhóm trễ hạn: `PAY_MAX` (13,4%), `PAY_1` (9,9%), `PAY_LATE_CONSECUTIVE` (7,5%), `PAY_LATE_2PLUS_COUNT` (6,1%), tổng cộng 36,8% mức độ quan trọng.
- **Dư nợ và hạn mức đứng tiếp theo:** `BILL_AMT1`, `LIMIT_BAL`, `BILL_DELTA`, `UTIL_2`, `UTIL_1` xếp hạng 5–10. Các biến số tiền trả `PAY_AMT1`, `PAY_AMT2`, `PAY_AMT6` xếp hạng 11–14.
- **Mức độ quan trọng tập trung hơn so với XGBoost:** Top 10 biến chiếm 60,8% và Top 20 chiếm 81,4% (XGBoost: 46% và 70%). Có 37/119 cột có mean(|SHAP|) = 0 (XGBoost: 30/119), chủ yếu là các mức hiếm của `PAY_x`.
- **Biến nhân khẩu học đóng góp thấp:** `EDUCATION` hạng 24, `MARRIAGE` hạng 26, `AGE_BIN` hạng 36/49, tổng cộng 3,1% (XGBoost: 4,4%).
- **Thứ hạng ổn định theo mẫu:** Spearman ≥ 0,998 với hai mẫu Valid khác và một mẫu Train, Top 10 trùng 10/10, thứ tự năm nhóm không đổi.

### 6.3. Mức độ quan trọng theo nhóm đặc trưng

| Nhóm | Tỷ trọng LightGBM | Hạng | Tỷ trọng XGBoost | Hạng | Tỷ lệ triệt tiêu (LightGBM) |
|---|---|---|---|---|---|
| Lịch sử trễ hạn | 48,7% | 1 | 38,5% | 1 | 19% |
| Mức sử dụng hạn mức | 18,7% | 2 | 16,7% | 3 | 15% |
| Dư nợ và biến động dư nợ | 16,4% | 3 | 15,4% | 4 | 27% |
| Hành vi trả nợ | 13,5% | 4 | 24,1% | 2 | 51% |
| Nhân khẩu học | 2,8% | 5 | 5,3% | 5 | 35% |

- Nhóm lịch sử trễ hạn chiếm gần một nửa tổng mức độ quan trọng và có tỷ lệ triệt tiêu thấp (các biến trong nhóm đóng góp cùng chiều).
- Nhóm hành vi trả nợ giảm từ hạng 2 xuống hạng 4 và có tỷ lệ triệt tiêu cao nhất (51%): các biến số tiền trả và tỷ lệ trả/dư nợ thường mang dấu ngược nhau trên cùng một khách hàng.
- Đối với reason codes, nhóm lịch sử trễ hạn sẽ xuất hiện thường xuyên hơn, nhóm hành vi trả nợ ít hơn so với kỳ vọng rút ra từ XGBoost.

### 6.4. Dạng quan hệ giữa đặc trưng và đóng góp SHAP

- **Các biến có ràng buộc đơn điệu cho đường trung vị tăng theo giá trị.** `PAY_LATE_CONSECUTIVE` tăng từ −0,09 (0 tháng) lên +0,42 (6 tháng liên tiếp); `PAY_LATE_2PLUS_COUNT` từ −0,08 lên +0,40; `PAY_MEAN` gần phẳng (−0,07 đến −0,04) khi ≤ 0 và tăng lên +0,24 ở mức cao nhất.
- **`PAY_MAX`: bậc thang.** Trung vị SHAP ≈ −0,18 khi `PAY_MAX` ≤ 0, +0,27 tại mức 1, +0,41 tại mức 2 và khoảng +0,5 từ mức 3 trở lên. Đường trung vị dao động ở các mức ≥ 4 do ít quan sát (≤ 21 khách hàng mỗi mức). Biên độ nhỏ hơn XGBoost (+0,9 đến +1,1 tại mức 2–3) do phần đóng góp của lịch sử trễ hạn được chia cho `PAY_LATE_*` và `PAY_MEAN`.
- **`UTIL_2`, `UTIL_1`: gần phẳng ở mức sử dụng dưới khoảng 0,4,** trung vị SHAP trong khoảng −0,07 đến −0,04; tăng dần lên +0,14 (`UTIL_1`) và +0,19 (`UTIL_2`) khi mức sử dụng tiến tới hạn mức.
- **`PAY_1` (không ràng buộc) vẫn không đơn điệu.** Trễ 2 tháng cho SHAP trung vị +0,81, trễ 3 tháng +0,54, trễ từ 4 tháng trở lên chỉ ≈ +0,03 (16 khách hàng). Mức −1 (+0,01) cao hơn mức 0 (−0,18).
- **`LIMIT_BAL`: giảm đều,** từ +0,15 (hạn mức 20.000) xuống −0,14 (460.000).
- **`BILL_AMT1` và `BILL_DELTA`: giảm mạnh ở vùng giá trị thấp, sau đó tăng nhẹ.** Dư nợ bằng 0 cho SHAP ≈ +0,22; dư nợ từ khoảng 18.000 trở lên cho SHAP quanh −0,08, tăng dần về khoảng 0 ở vùng dư nợ rất lớn. Ở cùng mức dư nợ, khách hàng có `UTIL_2` cao nhận SHAP cao hơn, nhưng cường độ tương tác thấp (0,011). `BILL_DELTA` có dạng tương tự (+0,14 ở vùng gần 0, thấp nhất −0,09).

### 6.5. Tương tác

- **Tương tác yếu hơn nhiều so với XGBoost.** Cặp mạnh nhất vẫn là `PAY_1` – `PAY_MAX` nhưng cường độ chỉ 0,048 (XGBoost: 0,233), nhỏ hơn nhiều so với hiệu ứng chính của `PAY_MAX` (0,29) và `PAY_1` (0,18).
- Các cặp tiếp theo đều thuộc nhóm trễ hạn (`PAY_1` – `PAY_LATE_CONSECUTIVE` 0,027; `PAY_MAX` – `PAY_MEAN` 0,022). Cặp liên nhóm mạnh nhất chỉ đạt 0,014 (`PAY_1` – `PAY_RATIO_MEAN`).
- Kết quả phù hợp với cấu hình cây nông (10 lá) và regularization mạnh của mô hình tuned. Đóng góp của từng nhóm vì vậy gần như cộng tính, thuận lợi cho việc diễn giải reason codes.

### 6.6. `PAY_1` và reason codes

Với 16 khách hàng có `PAY_1` ≥ 4 (0,5% mẫu), SHAP riêng của `PAY_1` ≈ +0,03 nhưng SHAP ròng của nhóm lịch sử trễ hạn đạt +1,42 đến +1,64 và PD trung bình 0,60–0,69. Phần rủi ro được ghi nhận qua `PAY_MAX` và `PAY_LATE_*`. Reason codes theo nhóm vì vậy không bị ảnh hưởng bởi điểm không đơn điệu của `PAY_1`. Ngược lại, Waterfall Plot theo biến gốc có thể hiển thị `PAY_1` gần 0 cho khách hàng trễ hạn nặng; Waterfall theo nhóm không gặp vấn đề này.

### 6.7. Đối chiếu với kết luận trên XGBoost mặc định

| Kết luận tại `03_shap.ipynb` / `03_shap_dependence.ipynb` | Trên LightGBM tuned + monotonic |
|---|---|
| Lịch sử trễ hạn quan trọng nhất; `PAY_MAX` hạng 1, `PAY_1` hạng 2 | Giữ nguyên |
| `PAY_LATE_*`, `PAY_MEAN` xếp hạng thấp (31–41) do trùng thông tin với `PAY_MAX` | Không giữ: hạng 3, 4 và 9 |
| Hành vi trả nợ là nhóm quan trọng thứ hai (24,1%) | Không giữ: hạng 4 (13,5%) |
| `BILL_AMT1` không đơn điệu, phụ thuộc mạnh vào mức sử dụng hạn mức | Giữ một phần: vẫn không đơn điệu, tương tác với `UTIL_2` yếu (0,011) |
| `PAY_1` – `PAY_MAX` là tương tác nổi bật (0,233) | Vẫn mạnh nhất nhưng yếu hơn khoảng 5 lần (0,048) |
| `PAY_1` không đơn điệu ở mức trễ ≥ 4 tháng | Giữ nguyên |
| Thứ hạng nhóm ổn định theo mẫu | Giữ nguyên |

Thứ hạng theo biến khác đáng kể giữa hai mô hình (Spearman 0,73; Top 10 trùng 4/10): SHAP phân bổ đóng góp giữa các biến tương quan cao theo cách riêng của từng mô hình. Nhóm đứng đầu (lịch sử trễ hạn) không đổi. Kết quả này củng cố thiết kế reason codes theo nhóm đặc trưng (`project_plan.md` mục 3.5e). Đề xuất xem xét loại `PAY_LATE_*` do đóng góp SHAP thấp (`03_shap.ipynb` mục 7.6) không còn phù hợp với mô hình cuối.

### 6.8. Giới hạn

- Giá trị SHAP ở thang margin của mô hình chưa hiệu chuẩn. Với Platt scaling $\text{PD} = \sigma(a\,m + b)$, giá trị trên thang logit(PD) bằng $a \cdot \phi_i$; thứ hạng và dạng quan hệ không đổi, giá trị hiển thị trên demo cần nhân hệ số $a$ sau khi calibration hoàn tất (Tuần 3 – T2).
- Phân tích áp dụng cho mô hình chính; mô hình đối chứng CatBoost tuned chưa được phân tích.
- Đường xu hướng kém tin cậy ở các mức hiếm (`PAY_MAX` ≥ 4, `PAY_1` ≥ 4, dư nợ trên 600.000).

### 6.9. Đầu vào cho các bước tiếp theo

- **So sánh mức đồng thuận (Tuần 3 – T2):** bảng nhóm của notebook này và `_vs_xgboost_by_group.csv` là đầu vào; phần còn thiếu là Logistic.
- **Tích hợp Streamlit (Tuần 3 – T3):** Waterfall theo nhóm phù hợp hơn Waterfall theo biến gốc (mục 6.6); giá trị SHAP nhân hệ số Platt $a$.
- **Reason codes và fairness (Tuần 3 – T3):** nhóm nhân khẩu học chiếm 2,8%; việc hiển thị nhóm này trong reason codes phụ thuộc kết quả fairness.

## Hình và bảng xuất ra

Lưu tại `reports/figures/` (không commit, sinh lại bằng cách chạy notebook), tiền tố `shap_lightgbm_monotone_`:

- `_summary.png` — Summary Plot (beeswarm), Top 20 cột.
- `_bar_by_variable.png`, `_importance_by_variable.csv` — mức độ quan trọng theo biến gốc.
- `_bar_by_group.png`, `_importance_by_group.csv` — mức độ quan trọng theo nhóm đặc trưng.
- `_dependence_top9.png` — Dependence Plot Top 9 biến.
- `_interaction_pairs.csv` — cường độ tương tác theo cặp biến.
- `_vs_xgboost_by_group.csv` — tỷ trọng và thứ hạng nhóm so với XGBoost mặc định.